# Test 2 of the Occlusion Analysis in ResNet18 (Encrypted Version)

This Jupyter Notebook consists of the tests for the three algorithms for the encrypted Occlusion Analysis on ResNet18.

ResNet18 is a neural network that was trained on the ImageNet dataset.

The three algorithms used are the sliding-window algorithm, the hierarchical algorithm, and the Monte Carlo algorithm. They use two parties for the inference, whereas Alice owns the network and encrypts it, and Bob owns the data and encrypts it.

The cells of the presets must be run before the algorithms of the occlusion analysis. The algorithms don't need to be executed in a specific order.

Please note that this code was used to test the metrics of sparsity, robustness and consistency. The description of the algorithms can be seen in the non-test Jupyter notebooks.

## Presets

The presets contain all the important imports, the definitions of the parties, a function to process ResNet18 to make it functional for safe inference and a function to process the picture so that it is suitable for the inference.

Here the specific definition of the parties is required. Alice has rank 0, and Bob has rank 1.

The function prepare_resnet() transforms the PyTorch model of the pretrained version of ResNet18 into an ONNX model and saves it. This step is required because of an issue with the normal conversion. There was an error with the Identity op with CrypTen and ONNX.

The function process_image takes the picture from a specific path and transforms it so it can be used in the inference with ResNet18. It saves the picture to a specific save path.

In [ ]:
# PyTorch and Torchvision
import torch
import torchvision.models as models
import torchvision.transforms as transforms

# Standard Libraries
from PIL import Image
import json
import requests
import os

# Third-Party Libraries
import numpy as np
import matplotlib.pyplot as plt
import onnx

# CrypTen
import crypten
import crypten.mpc as mpc
from crypten.nn import onnx_converter
import crypten.communicator as comm

# Initializing CrypTen
crypten.init()
torch.set_num_threads(1)

In [ ]:
# Definition of the two parties
ALICE = 0
BOB = 1

In [ ]:
def prepare_resnet18():
    """
    Takes the ResNet18 model and converts it into a ONNX-model
    Skips Identity-Operations that cause errors
    Saves the model to a file so it can be loaded and encrypted later
    """
    # The ResNet18 model
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.eval() # Put model in inference phase

    # Dummy Input for ResNet18 
    dummy_input = torch.randn(1, 3, 224, 224)  

    # Actual conversion from pytorch to ONNX
    onnx_path="resnet18.onnx"
    torch.onnx.export(
        model,
        dummy_input,
        "resnet18.onnx",
        input_names = ["input"],
        output_names = ["output"],
        do_constant_folding = True, #For optimization
        opset_version = 9,  #Supported Version for CrypTen
        dynamic_axes = {"input": {0: "batch"}},  
        
    )
    # Reads ONNX-file as a byte stream
    with open(onnx_path, "rb") as f:
        onnx_model_bytes = f.read()
    return onnx_model_bytes

onnx_model_bytes = prepare_resnet18()    

In [ ]:
# Helper functions
def calculate_gini_index(heatmap):
    """Calculate Gini index for heatmap sparsity measurement"""
    flattened = heatmap.flatten().abs()
    flattened += 1e-10  # to avoid division by zero
    flattened = torch.sort(flattened).values
    n = flattened.shape[0]
    index = torch.arange(1, n+1)
    return 1 - 2 * (torch.sum((flattened * (n - index + 0.5))) / (n * torch.sum(flattened)))

def calculate_sens_value(heatmap_clean, heatmap_noisy):
    """Calculate normalized SENS value using L2 norms"""
    diff = heatmap_clean - heatmap_noisy
    l2_diff = torch.linalg.norm(diff)
    l2_clean = torch.linalg.norm(heatmap_clean)
    return l2_diff / (l2_clean + 1e-10)  # to avoid division by zero

## The sliding window algorithm

This function implements the encrypted version of the sliding-window algorithm. 

First it loads the model, and Alice encrypts it. Bob loads his encrypted data as well. All of it is done without encryption. Since the model is trained on the ImageNet dataset, the picture needs to be of size 224x224, which is provided by the function in the preset.

Next, the parameters of the occlusion analysis are defined. The patch_size is the size of the patch that will be used to mask the regions of the picture. The patch_size doesn't need to fit in the picture entirely since there is edge treatment, but it needs to be smaller than the picture. The occluded_value is the value of the mask used for the occlusion. In the standard case, it is set to black. These two parameters can be changed, especially the size of the patch. The last parameter is the heatmap. It is the same size as the picture, and at the beginning it is black. Also, it is encrypted to Bob.

The code loops over every picture and computes the heatmap for the not-noisy picture and the noisy picture. Both of the heatmaps get saved and the Gini-Index of the clean heatmap and the sens between the clean and noisy heatmap get calculated. In the end, the results will be printed and statistics will be visualized. 

In [ ]:
save_dir = "heatmaps"
clean_images_dir = "/tmp/clean_images"
noisy_images_dir = "/tmp/noisy_images"
num_images = 10 # Number needs to be adjusted according to number of pictures

# Create directories
os.makedirs(save_dir, exist_ok=True)

# Load model an convert it
private_model = crypten.nn.from_onnx(onnx_model_bytes)

# Worldsize is 2 because of the two parties
@mpc.run_multiprocess(world_size=2)
def occlusion_analysis_enc():
    """
    Performs the standard case of occlusion analysis on Resnet18.
    The algorithm is an encrypted version of the sliding-window algorithm.
    The result is a heatmap.
    """

    # Encrypt the model from Alice
    private_model.encrypt(src = ALICE)

    results = []
    
    for i in range(num_images):

        # Load the picture and encrypt it with Bob
        data_enc = crypten.load_from_party(f'noisy_images/processed_{i}.pth', src = BOB).unsqueeze(0)
    
        data_noisy_enc = crypten.load_from_party(f'noisy_images/noisy_pic_{i}.pth', src = BOB).unsqueeze(0)
       
        # Parameters for Occlusion Analysis 
        # - patch size
        # - occluded_value
        # - encrypted heatmap
        patch_size = 112 # Can be changed
        occluded_value = 0 # Can be changed

        heatmaps = {}
        for mode, current_data in [('clean', data_enc), ('noisy', data_noisy_enc)]:
    
            # Computation of original output of the original picture
            with torch.no_grad(): # Allows quicker computation
                original_output = private_model(current_data)#.view(1, 784)) # View is needed for the input of the model

            url = "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json"
            class_idx = requests.get(url).json()
            def get_class_name(class_id):
                return class_idx[str(class_id)][1] 
        
            original_output_plain = original_output.get_plain_text()
            original_pred = original_output_plain.argmax().item() #index of the class with highest propability
            predicted_class_name = get_class_name(original_pred)       
            crypten.print(f'Image {i} {mode} - Actual prediction: {predicted_class_name}')
            
            heatmap_enc = crypten.cryptensor(torch.zeros(current_data.shape[2], current_data.shape[3]), src = BOB)
            
            num_it = 1 # Counter for number of iterations
        
            # Iterating over the picture
            for y in range(0, current_data.shape[2], patch_size):
                for x in range(0, current_data.shape[3], patch_size):
                    # Edge treatment:
                    # The normal patch size is used 
                    # Unless the patch doesn't entirely fit in the edge
                    # Then the patch_size is the image.size - the current pixel position
                    better_patch_size_y = min(patch_size, current_data.shape[2] - y)
                    better_patch_size_x = min(patch_size, current_data.shape[3] - x)
                    
                    # Used if patch would be too small
                    if better_patch_size_y < 1 or better_patch_size_x < 1:
                        continue
        
                    # Creating a mask the size of the picture with value one
                    # Marking the relevant place, setting it to the occluded value
                    # Encrypting the mask
                    mask = torch.ones(current_data.shape[0], current_data.shape[1], current_data.shape[2], current_data.shape[3])
                    mask[:, :, y : y + better_patch_size_y, x : x + better_patch_size_x] = occluded_value
                    mask_enc = crypten.cryptensor(mask, src = BOB)
        
                    # Putting the mask on the picture
                    occluded_enc = current_data * mask_enc
        
                    # Calculating the occluded_output
                    with torch.no_grad():
                        occluded_output = private_model(occluded_enc)
        
                    # Calculating the absolute difference of the original output and the occluded output
                    diff_enc = (original_output - occluded_output).abs().sum()
        
                    # Putting the calculated difference on the encrypted heatmap
                    # By creating a patch that has value one on the relevant region
                    # Only where the value is one the diff will be added to the heatmap
                    place_heatmap = torch.zeros(heatmap_enc.shape[0], heatmap_enc.shape[1])
                    place_heatmap[y : y  + better_patch_size_y, x : x + better_patch_size_x] = 1
                    heatmap_enc=heatmap_enc + diff_enc * crypten.cryptensor(place_heatmap, src = BOB)
        
                    crypten.print(f"\rNumber of iterations: {num_it}", end='', flush=True)
                    num_it += 1
        
            # Decrypting the heatmap         
            heatmap = heatmap_enc.get_plain_text()
           
            # Normalize heatmap
            heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
            heatmaps[mode] = heatmap_norm
            
            # Save heatmap
            torch.save(heatmap_norm, f'{save_dir}/heatmap_{mode}_{i}.pt')
            
            # Calculate metrics
            if mode == 'clean':
                gini = calculate_gini_index(heatmap_norm)
            results.append({
                'image_id': i,
                'mode': mode,
                'heatmap': heatmap_norm,
                'gini_index': gini.item(),
                'prediction': predicted_class_name
            })
                
            # Visualize heatmap
            plt.figure(figsize=(10, 5))
            plt.imshow(heatmap_norm, cmap='hot')
            plt.title(f"Heatmap {mode} Image {i} (Pred: {predicted_class_name})")
            plt.colorbar()
            plt.savefig(f'{save_dir}/heatmap_{mode}_{i}.png')
            plt.close()
            
        # Calculate SENS value between clean and noisy heatmaps
        sens = calculate_sens_value(heatmaps['clean'], heatmaps['noisy'])
        
        # Update results for clean image
        for res in results:
            if res['image_id'] == i and res['mode'] == 'clean':
                res['sens'] = sens.item()
                break
    
    # Print summary statistics
    clean_ginis = [res['gini_index'] for res in results if res['mode'] == 'clean']
    sens_values = [res['sens'] for res in results if 'sens' in res and res['mode'] == 'clean']
    id_values = [res['image_id'] for res in results if res['mode'] == 'clean']
    
    crypten.print("\nSummary Statistics:")
    crypten.print(f"Average Gini index (clean): {np.mean(clean_ginis):.4f} ± {np.std(clean_ginis):.4f}, Max:{np.max(clean_ginis):.4f}, Min:{np.min(clean_ginis):.4f}")
    crypten.print(f"Average SENS value: {np.mean(sens_values):.4f} ± {np.std(sens_values):.4f}, Max:{np.max(sens_values):.4f}, Min:{np.min(sens_values):.4f}")

    plt.figure(figsize=(10, 5))
    plt.plot(clean_ginis, 'o-')
    plt.show()
    
    plt.figure(figsize=(10, 5))
    plt.plot(sens_values, 'o-')
    plt.show()
    
occlusion_analysis_enc()

## The hierarchical occlusion algorithm

This function implements the encrypted version of the hierarchical occlusion algorithm. 

First it loads the model, and Alice encrypts it. Bob loads his encrypted data as well. All of it is done without encryption. Since the model is trained on the ImageNet dataset, the picture needs to be of size 224x224, which is provided by the function in the preset.

Next, the parameters of the occlusion analysis are defined. The initial_patch is the size of the patch that will be used to mask the regions of the picture and is the starting size of the patch. The initial_patch doesn't need to fit in the picture entirely since there is edge treatment, but it needs to be smaller than the picture. The initial_patch should be dividable by 2 to avoid artifacts in the heatmap. The minimal_patch is the size of the smallest patch, so it should be smaller than the initial patch. However, the minimal_patch cannot always be the minimal patch; e.g., if the patch_size is 6 and the minimal patch is 2, the smallest patch will be 3 to avoid artifacts within the heatmap. The occluded_value is the value of the mask used for the occlusion. In the standard case, it is set to black. The threshold is needed to make the decision if a patch should be divided or not, and it needs to be encrypted as well. It can be any number between 0 and 1. These four parameters can be changed, especially the size of the patch, the minimal patch and the threshold. The last parameter is the heatmap. It is the same size as the picture, and at the beginning it is black. Also, it is encrypted to Bob.

The code loops over every picture and computes the heatmap for the not-noisy picture and the noisy picture. Both of the heatmaps get saved and the Gini-Index of the clean heatmap and the sens between the clean and noisy heatmap get calculated. In the end, the results will be printed and statistics will be visualized. 

In [ ]:
save_dir = "heatmaps"
clean_images_dir = "/tmp/clean_images"
noisy_images_dir = "/tmp/noisy_images"
num_images = 10 # Number needs to be adjusted according to number of pictures

# Create directories
os.makedirs(save_dir, exist_ok=True)

# Load model an convert it
private_model = crypten.nn.from_onnx(onnx_model_bytes)

# Worldsize is 2 because of the two parties
@mpc.run_multiprocess(world_size=2)
def hierarchical_occlusion_enc(): 
    """
    Performs an encrypted hierarchical occlusion analysis on ResNet18.
    The algorithm identifies regions that are important for the model and partitions the patch.
    The result is a heatmap.
    """
    # Encrypt the model from Alice
    private_model.encrypt(src = ALICE)

    results = []
    
    for i in range(num_images):

        # Load the picture and encrypt it with Bob
        data_enc = crypten.load_from_party(f'noisy_images/processed_{i}.pth', src = BOB).unsqueeze(0)
    
        data_noisy_enc = crypten.load_from_party(f'noisy_images/noisy_pic_{i}.pth', src = BOB).unsqueeze(0)
    
        # Parameters for Occlusion Analysis 
        # - initial_patch
        # - minimal_patch
        # - occluded_value
        # - encrypted threshold
        # - encrypted heatmap
        initial_patch = 112 # Can be changed
        minimal_patch = 112 # Can be changed
        occluded_value = 0 # Can be changed
        threshold_enc = crypten.cryptensor(0.5, src = BOB) # Can be changed
        
        heatmaps = {}
        for mode, current_data in [('clean', data_enc), ('noisy', data_noisy_enc)]:
    
            # Computation of original output of the original picture
            with torch.no_grad(): # Allows quicker computation
                original_output = private_model(current_data)#.view(1, 784)) # View is needed for the input of the model

            url = "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json"
            class_idx = requests.get(url).json()
            def get_class_name(class_id):
                return class_idx[str(class_id)][1] 
        
            original_output_plain = original_output.get_plain_text()
            original_pred = original_output_plain.argmax().item() #index of the class with highest probability
            predicted_class_name = get_class_name(original_pred)       
            crypten.print(f'Image {i} {mode} - Actual prediction: {predicted_class_name}')
            
            heatmap_enc = crypten.cryptensor(torch.zeros(current_data.shape[2], current_data.shape[3]), src = BOB)
            num_it = 1 # Counter for number of iterations
            
            def patch_partition(starting_point, current_patch_size, heatmap):
                """
                Partitions the patch if the difference of the original output and occluded output is important.
                Args:
                    starting_point: top-left corner of the current patch
                    current_patch_size: Size of the current patch
                    heatmap: Contains the difference of the original output and occluded output
                Returns:
                    heatmap
                """    
                nonlocal num_it
                y, x = starting_point
        
                # Edge treatment:
                # The normal patch size is used 
                # Unless the patch doesn't entirely fit in the edge
                # Then the patch_size is the image.size - the current pixel position
                better_patch_size_y = min(current_patch_size, current_data.shape[2] - y)
                better_patch_size_x = min(current_patch_size, current_data.shape[3] - x)
        
                # Used if patch would be too small
                if better_patch_size_y < 1 or better_patch_size_x < 1:
                    return heatmap      
                      
                # Creating a mask the size of the picture with value one
                # Marking the relevant place, setting it to the occluded value
                # Encrypting the mask
                mask = torch.ones(current_data.shape[0], current_data.shape[1], current_data.shape[2], current_data.shape[3])
                mask[:, : , y : y + better_patch_size_y, x : x + better_patch_size_x] = occluded_value
                mask_enc = crypten.cryptensor(mask, src = BOB)
        
                # Putting the mask on the picture
                occluded_enc = current_data * mask_enc
        
                # Calculating the occluded_output
                with torch.no_grad():
                    occluded_output = private_model(occluded_enc)
        
                # Calculating the absolute difference of the original output and the occluded output
                diff_enc = (original_output - occluded_output).abs().sum()
        
                # Putting the calculated difference on the encrypted heatmap
                # By creating a patch that has value one on the relevant region
                # Only where the value is one the diff will be added to the heatmap
                place_heatmap = torch.zeros(heatmap.shape[0], heatmap.shape[1])
                place_heatmap[y : y  + better_patch_size_y, x : x + better_patch_size_x] = 1
                heatmap = heatmap + diff_enc * crypten.cryptensor(place_heatmap, src = BOB)
        
                # Calculates the difference but in the range 0 to 1
                comp = ((original_output.softmax(1) - occluded_output.softmax(1)).abs().sum())/ 2
        
                # Decision if the patch should be divided
                # Difference needs to be bigger than threshold 
                # and patch must be bigger than smallest patch
                # Decision if patch is bigger then the smallest patch needs to be encrypted
                differentiate = (comp > threshold_enc) * crypten.cryptensor((float(current_patch_size)/2.0 >= float(minimal_patch)))
                
                # If decision is true, divide the patch 
                if differentiate.get_plain_text().item(): # Needed to be decrypted due to too complicated logic for crypten.where
                    half_patch = current_patch_size // 2
                    # Calculate possible positions and set new positions
                    for poss_y in [0, half_patch]:
                        for poss_x in [0, half_patch]:
                            new_y = poss_y + y
                            new_x = poss_x + x
                            # Check if patch position is bigger than picture
                            # if not start recursion
                            if (new_y < current_data.shape[2] and new_x < current_data.shape[3]):
                                heatmap = patch_partition((new_y, new_x), half_patch, heatmap)
                
                crypten.print(f"\rNumber of iterations: {num_it}", end='', flush=True)
                num_it += 1
                
                return heatmap
            
            # Start of occlusion analysis
            for y in range(0, current_data.shape[2], initial_patch):
                for x in range(0, current_data.shape[3], initial_patch):
                    heatmap_enc = patch_partition((y,x), initial_patch, heatmap_enc)          
            
            # Decrypting the heatmap         
            heatmap = heatmap_enc.get_plain_text()
        
            # Normalize heatmap
            heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
            heatmaps[mode] = heatmap_norm
            
            # Save heatmap
            torch.save(heatmap_norm, f'{save_dir}/heatmap_{mode}_{i}.pt')
            
            # Calculate metrics
            if mode == 'clean':
                gini = calculate_gini_index(heatmap_norm)
            results.append({
                'image_id': i,
                'mode': mode,
                'heatmap': heatmap_norm,
                'gini_index': gini.item(),
                'prediction': predicted_class_name
            })
                
            # Visualize heatmap
            plt.figure(figsize=(10, 5))
            plt.imshow(heatmap_norm, cmap='hot')
            plt.title(f"Heatmap {mode} Image {i} (Pred: {predicted_class_name})")
            plt.colorbar()
            plt.savefig(f'{save_dir}/heatmap_{mode}_{i}.png')
            plt.close()
            
        # Calculate SENS value between clean and noisy heatmaps
        sens = calculate_sens_value(heatmaps['clean'], heatmaps['noisy'])
        
        # Update results for clean image
        for res in results:
            if res['image_id'] == i and res['mode'] == 'clean':
                res['sens'] = sens.item()
                break
    
    # Print summary statistics
    clean_ginis = [res['gini_index'] for res in results if res['mode'] == 'clean']
    sens_values = [res['sens'] for res in results if 'sens' in res and res['mode'] == 'clean']
    id_values = [res['image_id'] for res in results if res['mode'] == 'clean']
    
    crypten.print("\nSummary Statistics:")
    crypten.print(f"Average Gini index (clean): {np.mean(clean_ginis):.4f} ± {np.std(clean_ginis):.4f}, Max:{np.max(clean_ginis):.4f}, Min:{np.min(clean_ginis):.4f}")
    crypten.print(f"Average SENS value: {np.mean(sens_values):.4f} ± {np.std(sens_values):.4f}, Max:{np.max(sens_values):.4f}, Min:{np.min(sens_values):.4f}")

    plt.figure(figsize=(10, 5))
    plt.plot(clean_ginis, 'o-')
    plt.show()
    
    plt.figure(figsize=(10, 5))
    plt.plot(sens_values, 'o-')
    plt.show()

hierarchical_occlusion_enc()

## The Monte Carlo Occlusion Algorithm

This function implements the encrypted version of the Monte Carlo algorithm. 

First it loads the model, and Alice encrypts it. Bob loads his encrypted data as well. All of it is done without encryption. Since the model is trained on the ImageNet dataset, the picture needs to be of size 224x224, which is provided by the function in the preset.

Next, the parameters of the occlusion analysis are defined. The patch_size is the size of the patch that will be used to mask the regions of the picture. The patch_size doesn't need to fit in the picture entirely since there is edge treatment, but it needs to be smaller than the picture. The occluded_value is the value of the mask used for the occlusion. In the standard case, it is set to black. The num_simulations is the number of simulations that the algorithm will do. This number shouldn't be set too small to achieve good results. Note that it isn't guaranteed that all the regions in the heatmap will be covered. These three parameters can be changed, especially the size of the patch and the number of iterations. The last parameter is the heatmap. It is the same size as the picture, and at the beginning it is black. Also, it is encrypted to Bob.

The code loops over every picture and computes the heatmap for the not-noisy picture and the noisy picture. Both of the heatmaps get saved and the Gini-Index of the clean heatmap and the sens between the clean and noisy heatmap get calculated. In the end, the results will be printed and statistics will be visualized. 

In [ ]:
save_dir = "heatmaps"
clean_images_dir = "/tmp/clean_images"
noisy_images_dir = "/tmp/noisy_images"
num_images = 10 # Number needs to be adjusted according to number of pictures

# Create directories
os.makedirs(save_dir, exist_ok=True)

# Load model an convert it
private_model = crypten.nn.from_onnx(onnx_model_bytes)

# Worldsize is 2 because of the two parties
@mpc.run_multiprocess(world_size=2)
def monte_carlo_occlusion_analysis_enc():
    """
    Performs an encrypted Monte Carlo occlusion analysis on ResNet18.
    The algorithm sets the patch to random places on the picture and calculates the difference.
    The result is a statistical heatmap.
    """
    # Set seeds for results  
    torch.manual_seed(42)
    np.random.seed(42)

    results=[]
    
    # Encrypt the model from Alice
    private_model.encrypt(src = ALICE)

    for i in range(num_images):

        # Load the picture and encrypt it with Bob
        data_enc = crypten.load_from_party(f'noisy_images/processed_{i}.pth', src = BOB).unsqueeze(0)
    
        data_noisy_enc = crypten.load_from_party(f'noisy_images/noisy_pic_{i}.pth', src = BOB).unsqueeze(0)
    
        # Parameters for Occlusion Analysis 
        # - patch_size
        # - occluded_value
        # - num_simulations
        # - encrypted heatmap
        patch_size = 100 # Can be changed
        occluded_value = 0 # Can be changed
        num_simulations = 7 # Can be changed
        
        heatmaps = {}
        for mode, current_data in [('clean', data_enc), ('noisy', data_noisy_enc)]:
    
            # Computation of original output of the original picture
            with torch.no_grad(): # Allows quicker computation
                original_output = private_model(current_data)#.view(1, 784)) # View is needed for the input of the model

            url = "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json"
            class_idx = requests.get(url).json()
            def get_class_name(class_id):
                return class_idx[str(class_id)][1] 
        
            original_output_plain = original_output.get_plain_text()
            original_pred = original_output_plain.argmax().item() #index of the class with highest probability
            predicted_class_name = get_class_name(original_pred)       
            crypten.print(f'Image {i} {mode} - Actual prediction: {predicted_class_name}')
            
            heatmap_enc = crypten.cryptensor(torch.zeros(current_data.shape[2], current_data.shape[3]), src = BOB)
        
            num_it = 1 # Counter for number of iterations
        
            # Actual_sim represents the number of actual simulations
            # Tried_sim is the number of tried simulations (those who failed and those who didn't
            # Max_sim is the number of maximum tries to avoid endless loop
            actual_sim = 0
            tried_sim = 0
            max_sim = num_simulations * 2 # Can be changed
        
            # Calculate the actual possible positions for the patch
            # By dividing the possible positions by the patch_size
            poss_positions_y = (current_data.shape[2] - patch_size) // patch_size + 1
            poss_positions_x = (current_data.shape[3] - patch_size) // patch_size + 1
        
            while actual_sim < num_simulations and tried_sim < max_sim:
                # Chose a position for the patch
                position_patch_y = np.random.randint(0, poss_positions_y + 1)
                position_patch_x = np.random.randint(0, poss_positions_y + 1)
        
                # Calculate the positions with the patch_size
                y = position_patch_y * patch_size
                x = position_patch_x * patch_size
        
                # Edge treatment:
                # The normal patch size is used 
                # Unless the patch doesn't entirely fit in the edge
                # Then the patch_size is the image.size - the current pixel position
                better_patch_size_y = min(patch_size, current_data.shape[2] - y)
                better_patch_size_x = min(patch_size, current_data.shape[3] - x)
        
                # Used if patch would be too small
                if better_patch_size_y < 1 or better_patch_size_x < 1:
                        tried_sim += 1
                        continue
        
                # Creating a mask the size of the picture with value one
                # Marking the relevant place, setting it to the occluded value
                # Encrypting the mask
                mask = torch.ones(current_data.shape[0], current_data.shape[1], current_data.shape[2], current_data.shape[3])
                mask[:, : , y : y + better_patch_size_y, x : x + better_patch_size_x] = occluded_value
                mask_enc = crypten.cryptensor(mask, src = BOB)
        
                # Putting the mask on the picture
                occluded_enc = current_data * mask_enc
        
                # Calculating the occluded_output
                with torch.no_grad():
                    occluded_output = private_model(occluded_enc)
        
                # Calculating the absolute difference of the original output and the occluded output
                diff_enc = (original_output - occluded_output).abs().sum()
        
                # Putting the calculated difference on the encrypted heatmap
                # By creating a patch that has value one on the relevant region
                # Only where the value is one the diff will be added to the heatmap
                place_heatmap = torch.zeros(heatmap_enc.shape[0], heatmap_enc.shape[1])
                place_heatmap[y : y  + better_patch_size_y, x : x + better_patch_size_x] = 1
                heatmap_enc = heatmap_enc + diff_enc * crypten.cryptensor(place_heatmap, src = BOB)
        
                crypten.print(f"\rNumber of iterations: {num_it}", end='', flush=True)
                num_it += 1
                tried_sim += 1
                actual_sim += 1
        
            # Calculate the average influence
            heatmap = heatmap_enc.get_plain_text()/ actual_sim
        
            # Normalize heatmap
            heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
            heatmaps[mode] = heatmap_norm
            
            # Save heatmap
            torch.save(heatmap_norm, f'{save_dir}/heatmap_{mode}_{i}.pt')
            
            # Calculate metrics
            if mode == 'clean':
                gini = calculate_gini_index(heatmap_norm)
            results.append({
                'image_id': i,
                'mode': mode,
                'heatmap': heatmap_norm,
                'gini_index': gini.item(),
                'prediction': predicted_class_name
            })
                
            # Visualize heatmap
            plt.figure(figsize=(10, 5))
            plt.imshow(heatmap_norm, cmap='hot')
            plt.title(f"Heatmap {mode} Image {i} (Pred: {predicted_class_name})")
            plt.colorbar()
            plt.savefig(f'{save_dir}/heatmap_{mode}_{i}.png')
            plt.close()
            
        # Calculate SENS value between clean and noisy heatmaps
        sens = calculate_sens_value(heatmaps['clean'], heatmaps['noisy'])
        
        # Update results for clean image
        for res in results:
            if res['image_id'] == i and res['mode'] == 'clean':
                res['sens'] = sens.item()
                break
    
    # Print summary statistics
    clean_ginis = [res['gini_index'] for res in results if res['mode'] == 'clean']
    sens_values = [res['sens'] for res in results if 'sens' in res and res['mode'] == 'clean']
    id_values = [res['image_id'] for res in results if res['mode'] == 'clean']
    
    crypten.print("\nSummary Statistics:")
    crypten.print(f"Average Gini index (clean): {np.mean(clean_ginis):.4f} ± {np.std(clean_ginis):.4f}, Max:{np.max(clean_ginis):.4f}, Min:{np.min(clean_ginis):.4f}")
    crypten.print(f"Average SENS value: {np.mean(sens_values):.4f} ± {np.std(sens_values):.4f}, Max:{np.max(sens_values):.4f}, Min:{np.min(sens_values):.4f}")

    plt.figure(figsize=(10, 5))
    plt.plot(clean_ginis, 'o-')
    plt.show()
    
    plt.figure(figsize=(10, 5))
    plt.plot(sens_values, 'o-')
    plt.show()

monte_carlo_occlusion_analysis_enc()

This code is used for cleaning the files up.

In [ ]:
import os

filenames = ['/tmp/processed_image.pth']

for fn in filenames:
    if os.path.exists(fn): os.remove(fn)